In [ ]:
library(Seurat)
library(ggplot2)
library(pheatmap)
library(dplyr)
library(RColorBrewer)
library(reshape2)
library(clustree)
getwd()
dir.create("figures_10xMouse_PBMC")
dir.create("data_10xMouse_PBMC")
dataset_id <- "10xMouse_PBMC"
sample <- "10k"
mode <- "pseudobulk"
RESULTS_PATH_Stellarscope <- "/mnt/TEresults2/snakemake_results"

colorPBMC <- "#81B29A"


In [ ]:
# celltype annotation

celltypes <- read.table(paste0("data_", dataset_id,"/celltype_annotation.tsv"), sep = "\t")
colnames(celltypes) <- c("barcode","celltype")
rownames(celltypes) <- celltypes$barcode

PBMCcelltypeColors <- c("B_cells"="#6D5A5D",
                        "Dendritic Cells"="#C4B3AB", #"#E78063",#"#c492a7",
                        "Monocytes"="#81B29A",
                        "Platelets"="#e78063",
                        "T_cells"="#84B6D6",
                        "Natural Killers"= "#F2CC8F",
                        "Neutrophils"="#c492a7",
                        "Unknown" = "#3D405B")

In [ ]:

path <- paste0("/mnt/TEresults2/snakemake_results/results/stellarscope_out/", dataset_id, "/", sample, "/", mode, "/")
STAR_path <- paste0("/mnt/TEresults/snakemake_results/results/STARoutdir/",dataset_id, "/", sample, "/best_Solo.out/Gene")

filteredBarcodes <- read.table(paste0(STAR_path, "/filtered/barcodes.tsv"))$V1 # read barcodes seleced by STARsolo
# remove cells annotated as multiplets by 10x
mouse_assignment <- read.csv("data_10xMouse_PBMC/SC3_v3_NextGem_DI_CellPlex_Mouse_PBMC_10K_Multiplex_multiplexing_analysis_assignment_confidence_table.csv")
mouse_assignment$Barcodes <- sapply(strsplit(mouse_assignment$Barcodes, split="-"), "[", 1)
multiplets <- mouse_assignment$Barcodes[mouse_assignment$Assignment=="Multiplet"]
filteredBarcodes <- setdiff(filteredBarcodes, multiplets)

TEmatrix <- Seurat::ReadMtx(mtx = paste0(path, sample, "_", mode, "-TE_counts.mtx"), 
                              cells = paste0(path, sample, "_", mode, "-barcodes.tsv"), 
                              features = paste0(path, sample, "_", mode, "-features.tsv"), 
                              feature.column = 1) # read matrix
TEmatrix <- TEmatrix[setdiff(rownames(TEmatrix),"__no_feature"),filteredBarcodes]

nCells <- ncol(TEmatrix)
thrMinCells <- round(nCells * 0.05)

# create Seurat object 
objTE_stellarscope <- Seurat::CreateSeuratObject(TEmatrix, project = "PBMC", 
                          min.cells = thrMinCells, min.features = 20)

In [ ]:
objTE_stellarscope

In [ ]:
annotation_stellarscope <- read.table("annotation/annotation_stellarscope.tsv") # generated with annotation_scripts/create_annotations_mouse.Rmd
head(annotation_stellarscope)

In [ ]:
# Remove some classes of TEs

classesToExclude <- c("Other", "Satellite", "Unknown", "RNA")

stellarscopeTEs <- Features(objTE_stellarscope)

TEsToKeep <- annotation_stellarscope[! annotation_stellarscope$class %in% classesToExclude, ]$stellarscopeID

length(setdiff(stellarscopeTEs, TEsToKeep))/ length(stellarscopeTEs) * 100 # percentage removed

objTE_stellarscope <- objTE_stellarscope[intersect(stellarscopeTEs, TEsToKeep),]

In [ ]:
feature_metadata <- annotation_stellarscope[match(Features(objTE_stellarscope), annotation_stellarscope$stellarscopeID),]

head(feature_metadata)

In [ ]:
options(repr.plot.width=7, repr.plot.height=5)


objTE_stellarscope@meta.data$nCount_TE <- objTE_stellarscope@meta.data$nCount_RNA 
objTE_stellarscope@meta.data$nFeature_TE <- objTE_stellarscope@meta.data$nFeature_RNA 
# Visualize QC metrics as a violin plot
VlnPlot(objTE_stellarscope, features = c("nCount_TE"), ncol = 1, 
        cols = colorPBMC, pt.size = 0) + theme(text=element_text(size=17))

ggsave(paste0("figures_",dataset_id,"/nCountTE_violin_stellarscope_",mode,".png"), device='png',dpi=600)
ggsave(paste0("figures_",dataset_id,"/nCountTE_violin_stellarscope_",mode,".pdf"), device='pdf')

VlnPlot(objTE_stellarscope, features = c("nFeature_TE"), ncol = 1, 
        cols = colorPBMC, pt.size = 0) + theme(text=element_text(size=17))
ggsave(paste0("figures_",dataset_id,"/nFeatureTE_violin_stellarscope_",mode,".png"), device='png',dpi=600)
ggsave(paste0("figures_",dataset_id,"/nFeatureTE_violin_stellarscope_",mode,".pdf"), device='pdf')


In [ ]:

objTE_stellarscope <- JoinLayers(objTE_stellarscope)

objTE_stellarscope <- NormalizeData(objTE_stellarscope, normalization.method = "LogNormalize", scale.factor = 10000)

objTE_stellarscope <- FindVariableFeatures(objTE_stellarscope, selection.method = "vst", nfeatures = 4000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(objTE_stellarscope), 10)

# plot variable features with and without labels
plot1 <- VariableFeaturePlot(objTE_stellarscope)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

In [ ]:

gc()
all.genes <- rownames(objTE_stellarscope)
objTE_stellarscope <- ScaleData(objTE_stellarscope) # on hvgs


In [ ]:

objTE_stellarscope <- RunPCA(objTE_stellarscope, features = VariableFeatures(object = objTE_stellarscope))

DimPlot(objTE_stellarscope, reduction = "pca") + NoLegend()

ElbowPlot(objTE_stellarscope)


In [ ]:
objTE_stellarscope <- FindNeighbors(objTE_stellarscope, dims = 1:10, k.param = 20)
objTE_stellarscope <- FindClusters(objTE_stellarscope, resolution = 1.2)
objTE_stellarscope <- RunUMAP(objTE_stellarscope, dims = 1:10)
DimPlot(objTE_stellarscope, reduction = "umap")


In [ ]:
FeaturePlot(objTE_stellarscope, reduction = "umap", features = "nCount_TE", pt.size = 0.5) + 
  theme_void() +
  theme(text=element_text(size=20))

In [ ]:
objTE_stellarscope$celltype <- celltypes$celltype[match(Cells(objTE_stellarscope), celltypes$barcode)]

In [ ]:
table(objTE_stellarscope$celltype)
sum(is.na(objTE_stellarscope$celltype))

In [ ]:
DimPlot(objTE_stellarscope, reduction = "umap", group.by="celltype")

In [ ]:

saveRDS(objTE_stellarscope, paste0("data_", dataset_id, "/stellarscope_", dataset_id, "_", "seuratObj.RDS"))